In [2]:
import os
from google import genai
from dotenv import load_dotenv

load_dotenv()

# The new SDK uses a Client object
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

try:
    response = client.models.generate_content(
        model='gemini-2.5-flash', # Or 'gemini-3-flash-preview'
        contents="Hello! Are you online?"
    )
    print(response.text)
except Exception as e:
    print(f"Error: {e}")

Hello! Yes, I am online and ready to help. How can I assist you today?


In [3]:
import os
from google import genai
from dotenv import load_dotenv

# Load the API key from your .env file
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

# Initialize the Gemini Client
client = genai.Client(api_key=api_key)

def get_completion(prompt, model="gemini-2.5-flash"):
    """
    A drop-in replacement for the OpenAI-style completion function 
    using the Google GenAI SDK.
    """
    try:
        # Gemini uses 'contents' instead of 'messages' for simple prompts
        response = client.models.generate_content(
            model=model,
            contents=prompt,
            config={
                'temperature': 0, # Keeps responses deterministic
            }
        )
        return response.text
    except Exception as e:
        return f"Error encountered: {str(e)}"

# Example Usage:
# result = get_completion("Diagnose a car engine clicking sound.")
# print(result)

In [4]:
# Call the function with your prompt
result = get_completion("What is the capital of France?")

# Print the final output
print(f"Assistant: {result}")

Assistant: The capital of France is **Paris**.


In [11]:
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

def get_completion_from_messages(messages, 
                                 model="gemini-2.5-flash", 
                                 temperature=1):
    # 1. Separate the 'system' content from the conversation history
    # We find the system message and take its content
    system_message = next((m['content'] for m in messages if m['role'] == 'system'), None)
    
    # 2. Filter the messages to only include 'user' and 'model' (assistant)
    # Gemini requires the 'parts' and 'text' keys
    history = []
    for m in messages:
        if m['role'] != 'system':
            # Map 'assistant' to 'model' for Gemini compatibility
            role = 'model' if m['role'] == 'assistant' else 'user'
            history.append({
                'role': role,
                'parts': [{'text': m['content']}]
            })

    # 3. Call the API with the proper structure
    response = client.models.generate_content(
        model=model,
        contents=history,
        config=types.GenerateContentConfig(
            system_instruction=system_message,
            temperature=temperature
        )
    )
    return response.text

# --- RUNNING THE CODE ---
messages = [  
    {'role':'system', 'content': "You are an assistant who responds in the style of Dr Seuss."},    
    {'role':'user', 'content': "write me a very short poem about a happy carrot"},  
] 

response = get_completion_from_messages(messages)
print(response)

Oh, the carrot, so snappy and bright!
He wiggled with giggles, a joyful delight!
From his green little top to his orangey toes,
"Hooray for the day!" was the song that he chose!


In [12]:
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

def get_completion_and_token_count(messages, 
                                   model="gemini-2.5-flash", 
                                   temperature=0, 
                                   max_output_tokens=500):
    """
    Returns the AI response text along with a detailed token usage dictionary.
    """
    # 1. Handle System Instruction and Chat History
    system_instr = next((m['content'] for m in messages if m['role'] == 'system'), None)
    
    formatted_contents = []
    for m in messages:
        if m['role'] != 'system':
            role = 'model' if m['role'] == 'assistant' else 'user'
            formatted_contents.append({
                'role': role, 
                'parts': [{'text': m['content']}]
            })

    # 2. Call Gemini
    response = client.models.generate_content(
        model=model,
        contents=formatted_contents,
        config=types.GenerateContentConfig(
            system_instruction=system_instr,
            temperature=temperature,
            max_output_tokens=max_output_tokens
        )
    )

    # 3. Extract Token Usage (The Metadata)
    usage = response.usage_metadata
    
    token_dict = {
        'prompt_tokens': usage.prompt_token_count,
        'completion_tokens': usage.candidates_token_count,
        'total_tokens': usage.total_token_count,
    }

    return response.text, token_dict

# --- Example Execution ---
messages = [
    {'role': 'system', 'content': 'You are a Senior Automotive Engineer.'},
    {'role': 'user', 'content': 'Explain the function of a turbocharger in 2 sentences.'}
]

content, tokens = get_completion_and_token_count(messages)

print(f"Response: {content}")
print(f"Usage: {tokens}")

Response: A turbocharger uses exhaust gases to spin a turbine, which in turn drives a compressor to force more air into the engine's cylinders. This increased air density allows for more fuel to be burned, significantly boosting engine power and efficiency.
Usage: {'prompt_tokens': 21, 'completion_tokens': 47, 'total_tokens': 134}
